# 🌧️ Predicción de Precipitación con TNN - Dataset JUN
**Modelos:** Predicción a 1h, 3h y 6h  
**Arquitectura:** Temporal Neural Network (Conv1D)  
**Resampleo:** 15 min → 1 hora (promedio para variables continuas, suma para lluvia)

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Dropout, Flatten, MaxPooling1D, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print(f'TensorFlow version: {tf.__version__}')
print(f'Pandas version: {pd.__version__}')

## 2. Carga del Dataset

In [ ]:
# ============================================================
# AJUSTA ESTA RUTA AL ARCHIVO CSV DE TU DATASET JUN
# ============================================================
FILE_PATH = '../Datasets/JUN_consolid_f15.csv'   # <-- cambia aquí

df = pd.read_csv(
    FILE_PATH,
    parse_dates=['TIMESTAMP'],
    index_col='TIMESTAMP',
    na_values=['NA', 'NaN', '']
)

print(f'Shape original: {df.shape}')
print(f'Rango de fechas: {df.index.min()} → {df.index.max()}')
df.head(8)

## 3. Exploración del Dataset (15 min)

In [ ]:
print('=== Tipos de datos ===')
print(df.dtypes)
print('\n=== Estadísticas descriptivas ===')
df.describe()

In [ ]:
print('=== Valores nulos por columna ===')
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(null_pct[null_pct > 0])

In [ ]:
# Visualización de la serie de precipitación a 15 min
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

axes[0].plot(df.index, df['Rain_mm_Tot'], color='steelblue', linewidth=0.8)
axes[0].set_title('Precipitación (15 min) - Rain_mm_Tot')
axes[0].set_ylabel('mm')

axes[1].plot(df.index, df['AirTC_Avg'], color='tomato', linewidth=0.8)
axes[1].set_title('Temperatura del Aire - AirTC_Avg')
axes[1].set_ylabel('°C')

axes[2].plot(df.index, df['RH_Avg'], color='mediumseagreen', linewidth=0.8)
axes[2].set_title('Humedad Relativa - RH_Avg')
axes[2].set_ylabel('%')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig('series_15min.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Resampleo de 15 min → 1 Hora

> **Regla:**  
> - `Rain_mm_Tot` → **suma** (acumulación de lluvia en la hora)  
> - Resto de variables → **media** (promedio de los 4 registros)

In [ ]:
# Columna objetivo de precipitación
RAIN_COL = 'Rain_mm_Tot'

# Separar columnas por tipo de agregación
other_cols = [c for c in df.columns if c != RAIN_COL]

# Resampleo
df_mean = df[other_cols].resample('1H').mean()
df_rain = df[[RAIN_COL]].resample('1H').sum()

df_hourly = pd.concat([df_mean, df_rain], axis=1)

print(f'Shape 15 min : {df.shape}')
print(f'Shape 1 hora : {df_hourly.shape}')
print(f'\nPrimeros registros horarios:')
df_hourly.head(6)

In [ ]:
# Verificar suma de lluvia: total 15min vs total horario
total_15min = df[RAIN_COL].sum()
total_1h    = df_hourly[RAIN_COL].sum()
print(f'Total lluvia (15 min): {total_15min:.4f} mm')
print(f'Total lluvia (1 hora): {total_1h:.4f} mm')
print(f'Diferencia           : {abs(total_15min - total_1h):.6f} mm  ✅')

In [ ]:
# Distribución de lluvia horaria
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(df_hourly.index, df_hourly[RAIN_COL], color='steelblue', linewidth=0.8)
axes[0].set_title('Precipitación Horaria (acumulada por hora)')
axes[0].set_ylabel('mm/h')

axes[1].hist(df_hourly[RAIN_COL], bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Distribución de Precipitación Horaria')
axes[1].set_xlabel('mm/h')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig('distribucion_lluvia_1h.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Horas con lluvia > 0 : {(df_hourly[RAIN_COL] > 0).sum()} / {len(df_hourly)}')

## 5. Preprocesamiento

In [ ]:
FEATURES = [
    'SlrkW_Avg',    # 3.3%  - Radiación solar
    'SlrMJ_Tot',    # 3.3%  - Energía solar acumulada
    'WS_ms_Avg',    # 3.4%  - Velocidad viento
    'NR_Wm2_Avg',   # 9.8%  - Radiación neta
    'VW',           # 2.8%  - Volumetric Water Content (humedad suelo)
    'VW_2',         # 2.7%  - Humedad suelo sensor 2
    'VW_3',         # 2.7%  - Humedad suelo sensor 3
    'WindDir',      # 16.8% - Dirección del viento
    'Rain_mm_Tot'   # 0.0%  - Target
]

data = df_hourly[FEATURES].copy()
data = data.interpolate(method='linear', limit=6)
data = data.dropna()

# Features temporales cíclicas
data['hour_sin']  = np.sin(2 * np.pi * data.index.hour / 24)
data['hour_cos']  = np.cos(2 * np.pi * data.index.hour / 24)
data['month_sin'] = np.sin(2 * np.pi * data.index.month / 12)
data['month_cos'] = np.cos(2 * np.pi * data.index.month / 12)

# Lags de lluvia
data['Rain_lag1'] = data['Rain_mm_Tot'].shift(1)
data['Rain_lag2'] = data['Rain_mm_Tot'].shift(2)
data['Rain_lag3'] = data['Rain_mm_Tot'].shift(3)

data = data.dropna()

print(f'Shape final: {data.shape}')
print(f'Features ({len(data.columns)}): {data.columns.tolist()}')

In [ ]:
plt.figure(figsize=(10, 7))
corr = data.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5)
plt.title('Matriz de Correlación - Variables Horarias')
plt.tight_layout()
plt.savefig('correlacion.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Funciones Auxiliares para los 3 Modelos TNN

In [ ]:
def create_sequences(data_scaled, target_col_idx, look_back, horizon):
    """
    Crea ventanas deslizantes.
    
    Parameters
    ----------
    data_scaled   : np.array  - datos normalizados
    target_col_idx: int       - índice de la columna objetivo (Rain)
    look_back     : int       - pasos temporales de entrada
    horizon       : int       - pasos a predecir (1, 3 o 6 horas)
    
    Returns
    -------
    X : (samples, look_back, features)
    y : (samples, horizon)  — predicción multi-step
    """
    X, y = [], []
    n = len(data_scaled)
    for i in range(n - look_back - horizon + 1):
        X.append(data_scaled[i : i + look_back, :])           # todas las features
        y.append(data_scaled[i + look_back : i + look_back + horizon, target_col_idx])  # solo lluvia
    return np.array(X), np.array(y)


def build_tnn_model(look_back, n_features, horizon,
                    filters1=64, filters2=128, kernel_size=3, dropout=0.2, lr=1e-3):
    """Construye el modelo TNN (Conv1D) para predicción multi-step."""
    model = Sequential([
        # Primera capa convolucional
        Conv1D(filters=filters1, kernel_size=kernel_size, activation='relu',
               padding='same', input_shape=(look_back, n_features)),
        BatchNormalization(),
        Dropout(dropout),
        
        # Segunda capa convolucional
        Conv1D(filters=filters2, kernel_size=kernel_size, activation='relu',
               padding='same'),
        BatchNormalization(),
        Dropout(dropout),
        
        # Tercera capa convolucional
        Conv1D(filters=filters2, kernel_size=kernel_size, activation='relu',
               padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(dropout),
        
        # Aplanar y capas densas
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(dropout),
        Dense(64, activation='relu'),
        Dropout(dropout),
        Dense(horizon)          # salida: 'horizon' pasos
    ])
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='mse',
        metrics=['mae']
    )
    return model


def inverse_rain(y_scaled, scaler, target_col_idx, n_features):
    """Desnormaliza solo la columna de lluvia."""
    dummy = np.zeros((y_scaled.shape[0] * y_scaled.shape[1], n_features))
    flat  = y_scaled.flatten()
    dummy[:, target_col_idx] = flat
    return scaler.inverse_transform(dummy)[:, target_col_idx].reshape(y_scaled.shape)


def evaluate_model(y_true, y_pred, horizon_label):
    """Calcula y muestra métricas de evaluación."""
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    rmse = np.sqrt(mean_squared_error(y_true_f, y_pred_f))
    mae  = mean_absolute_error(y_true_f, y_pred_f)
    r2   = r2_score(y_true_f, y_pred_f)
    print(f'\n📊 Métricas - Horizonte {horizon_label}')
    print(f'   RMSE : {rmse:.4f} mm')
    print(f'   MAE  : {mae:.4f} mm')
    print(f'   R²   : {r2:.4f}')
    return {'horizon': horizon_label, 'RMSE': rmse, 'MAE': mae, 'R2': r2}


print('✅ Funciones definidas correctamente')

## 7. Normalización y División de Datos

In [ ]:
# ============================================================
# PARÁMETROS GLOBALES - Ajusta si es necesario
# ============================================================
LOOK_BACK  = 24    # Ventana de entrada: 24 horas previas
TRAIN_FRAC = 0.70  # 70% train, 15% val, 15% test
VAL_FRAC   = 0.15
EPOCHS     = 100
BATCH_SIZE = 32
PATIENCE   = 15

# Índice de la columna Rain en el array
rain_idx = list(data.columns).index(RAIN_COL)
n_features = data.shape[1]
print(f'Índice de Rain: {rain_idx}  |  Num. features: {n_features}')

# Normalización (fit solo en train para evitar data leakage)
n_total = len(data)
n_train = int(n_total * TRAIN_FRAC)
n_val   = int(n_total * VAL_FRAC)

scaler = MinMaxScaler(feature_range=(0, 1))
data_np = data.values.astype(np.float32)

scaler.fit(data_np[:n_train])
data_scaled = scaler.transform(data_np)

print(f'\nTamaño total  : {n_total}')
print(f'Train         : {n_train}')
print(f'Validation    : {n_val}')
print(f'Test          : {n_total - n_train - n_val}')

---
## 8. MODELO 1 — Predicción a 1 Hora

In [ ]:
HORIZON_1H = 1

X_1h, y_1h = create_sequences(data_scaled, rain_idx, LOOK_BACK, HORIZON_1H)

# División manteniendo orden temporal
X_tr1, y_tr1 = X_1h[:n_train], y_1h[:n_train]
X_vl1, y_vl1 = X_1h[n_train:n_train+n_val], y_1h[n_train:n_train+n_val]
X_ts1, y_ts1 = X_1h[n_train+n_val:], y_1h[n_train+n_val:]

print(f'Train: {X_tr1.shape}  |  Val: {X_vl1.shape}  |  Test: {X_ts1.shape}')

In [ ]:
model_1h = build_tnn_model(LOOK_BACK, n_features, HORIZON_1H)
model_1h.summary()

In [ ]:
callbacks_1h = [
    EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
]

history_1h = model_1h.fit(
    X_tr1, y_tr1,
    validation_data=(X_vl1, y_vl1),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_1h,
    verbose=1
)

In [ ]:
# Curva de pérdida
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_1h.history['loss'],     label='Train Loss')
ax.plot(history_1h.history['val_loss'], label='Val Loss')
ax.set_title('Curva de Entrenamiento - Modelo TNN 1h')
ax.set_xlabel('Época'); ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout()
plt.savefig('loss_1h.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Predicción y desnormalización
pred_1h_scaled = model_1h.predict(X_ts1)
pred_1h = inverse_rain(pred_1h_scaled, scaler, rain_idx, n_features)
true_1h = inverse_rain(y_ts1,          scaler, rain_idx, n_features)

# Clip negativos (lluvia no puede ser negativa)
pred_1h = np.clip(pred_1h, 0, None)

metrics_1h = evaluate_model(true_1h, pred_1h, '1 Hora')

In [ ]:
# Visualización predicción vs real
n_show = min(200, len(true_1h))
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(true_1h[:n_show, 0],  label='Real',       linewidth=1.2)
ax.plot(pred_1h[:n_show, 0],  label='Predicción', linewidth=1.2, linestyle='--')
ax.set_title(f'Precipitación Real vs Predicha — Horizonte 1h (primeras {n_show} muestras de test)')
ax.set_ylabel('mm/h'); ax.legend()
plt.tight_layout()
plt.savefig('pred_vs_real_1h.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9. MODELO 2 — Predicción a 3 Horas

In [ ]:
HORIZON_3H = 3

X_3h, y_3h = create_sequences(data_scaled, rain_idx, LOOK_BACK, HORIZON_3H)

X_tr3, y_tr3 = X_3h[:n_train], y_3h[:n_train]
X_vl3, y_vl3 = X_3h[n_train:n_train+n_val], y_3h[n_train:n_train+n_val]
X_ts3, y_ts3 = X_3h[n_train+n_val:], y_3h[n_train+n_val:]

print(f'Train: {X_tr3.shape}  |  Val: {X_vl3.shape}  |  Test: {X_ts3.shape}')

In [ ]:
model_3h = build_tnn_model(LOOK_BACK, n_features, HORIZON_3H,
                            filters1=64, filters2=128, dropout=0.2)

callbacks_3h = [
    EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
]

history_3h = model_3h.fit(
    X_tr3, y_tr3,
    validation_data=(X_vl3, y_vl3),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_3h,
    verbose=1
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_3h.history['loss'],     label='Train Loss')
ax.plot(history_3h.history['val_loss'], label='Val Loss')
ax.set_title('Curva de Entrenamiento - Modelo TNN 3h')
ax.set_xlabel('Época'); ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout()
plt.savefig('loss_3h.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
pred_3h_scaled = model_3h.predict(X_ts3)
pred_3h = np.clip(inverse_rain(pred_3h_scaled, scaler, rain_idx, n_features), 0, None)
true_3h = inverse_rain(y_ts3, scaler, rain_idx, n_features)

metrics_3h = evaluate_model(true_3h, pred_3h, '3 Horas')

In [ ]:
# Comparar cada paso del horizonte
fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for i, ax in enumerate(axes):
    n_show = min(200, len(true_3h))
    ax.plot(true_3h[:n_show, i],  label='Real')
    ax.plot(pred_3h[:n_show, i],  label='Predicción', linestyle='--')
    ax.set_title(f'Paso +{i+1}h')
    ax.set_ylabel('mm/h'); ax.legend()
plt.suptitle('Predicción a 3 Horas — Comparación por Paso Temporal', y=1.01)
plt.tight_layout()
plt.savefig('pred_vs_real_3h.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 10. MODELO 3 — Predicción a 6 Horas

In [ ]:
HORIZON_6H = 6

X_6h, y_6h = create_sequences(data_scaled, rain_idx, LOOK_BACK, HORIZON_6H)

X_tr6, y_tr6 = X_6h[:n_train], y_6h[:n_train]
X_vl6, y_vl6 = X_6h[n_train:n_train+n_val], y_6h[n_train:n_train+n_val]
X_ts6, y_ts6 = X_6h[n_train+n_val:], y_6h[n_train+n_val:]

print(f'Train: {X_tr6.shape}  |  Val: {X_vl6.shape}  |  Test: {X_ts6.shape}')

In [ ]:
# Para horizonte más largo, arquitectura ligeramente más robusta
model_6h = build_tnn_model(LOOK_BACK, n_features, HORIZON_6H,
                            filters1=128, filters2=256, dropout=0.25)

callbacks_6h = [
    EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
]

history_6h = model_6h.fit(
    X_tr6, y_tr6,
    validation_data=(X_vl6, y_vl6),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_6h,
    verbose=1
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_6h.history['loss'],     label='Train Loss')
ax.plot(history_6h.history['val_loss'], label='Val Loss')
ax.set_title('Curva de Entrenamiento - Modelo TNN 6h')
ax.set_xlabel('Época'); ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout()
plt.savefig('loss_6h.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
pred_6h_scaled = model_6h.predict(X_ts6)
pred_6h = np.clip(inverse_rain(pred_6h_scaled, scaler, rain_idx, n_features), 0, None)
true_6h = inverse_rain(y_ts6, scaler, rain_idx, n_features)

metrics_6h = evaluate_model(true_6h, pred_6h, '6 Horas')

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(14, 18))
for i, ax in enumerate(axes):
    n_show = min(200, len(true_6h))
    ax.plot(true_6h[:n_show, i],  label='Real')
    ax.plot(pred_6h[:n_show, i],  label='Predicción', linestyle='--')
    ax.set_title(f'Paso +{i+1}h')
    ax.set_ylabel('mm/h'); ax.legend()
plt.suptitle('Predicción a 6 Horas — Comparación por Paso Temporal', y=1.01)
plt.tight_layout()
plt.savefig('pred_vs_real_6h.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 11. Comparación Final de Modelos TNN

In [ ]:
results = pd.DataFrame([metrics_1h, metrics_3h, metrics_6h])
results = results.set_index('horizon')
print('\n=== RESUMEN COMPARATIVO TNN ===')
print(results.to_string())
results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, ['RMSE', 'MAE', 'R2']):
    colors = ['#2196F3', '#4CAF50', '#FF9800']
    bars = ax.bar(results.index, results[metric], color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{metric} por Horizonte')
    ax.set_xlabel('Horizonte de predicción')
    ax.set_ylabel(metric)
    for bar, val in zip(bars, results[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Comparación de Métricas — TNN (1h / 3h / 6h)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('comparacion_modelos_tnn.png', dpi=120, bbox_inches='tight')
plt.show()

## 12. Guardado de Modelos

In [ ]:
import joblib

model_1h.save('tnn_model_1h.keras')
model_3h.save('tnn_model_3h.keras')
model_6h.save('tnn_model_6h.keras')

joblib.dump(scaler, 'scaler_JUN.pkl')

print('✅ Modelos TNN y scaler guardados correctamente.')
print('   tnn_model_1h.keras')
print('   tnn_model_3h.keras')
print('   tnn_model_6h.keras')
print('   scaler_JUN.pkl')

---
## 📝 Resumen Metodológico - TNN

| Etapa | Detalle |
|---|---|
| **Resampleo** | 15 min → 1h: media para variables continuas, **suma** para lluvia |
| **Features** | SlrkW, SlrMJ, WS_ms, NR_Wm2, VW (1-3), WindDir + Rain + features temporales + lags |
| **Ventana** | 24 horas previas (look_back = 24) |
| **Horizonte** | Multi-step: 1h / 3h / 6h |
| **Arquitectura** | Conv1D(64) → BN → Dropout → Conv1D(128) → BN → Dropout → Conv1D(128) → BN → MaxPool → Dropout → Flatten → Dense(128) → Dense(64) → Dense(horizon) |
| **Normalización** | MinMaxScaler [0,1], fit solo en train |
| **División** | 70% train / 15% val / 15% test (orden temporal) |
| **Callbacks** | EarlyStopping + ReduceLROnPlateau |
| **Post-proceso** | clip(0) para evitar precipitaciones negativas |

### Diferencias clave vs GRU:
- **TNN usa Conv1D**: Captura patrones locales temporales mediante convoluciones 1D
- **Más parámetros**: Arquitectura más profunda con múltiples capas convolucionales
- **BatchNormalization**: Estabiliza el entrenamiento
- **MaxPooling**: Reduce dimensionalidad y captura características más robustas
- **Mejor para patrones locales**: Las convoluciones son eficientes detectando patrones en ventanas temporales pequeñas